# Bimodality fix attempt: joint per-sample activity regularizer

Closes review item #4. `rate_kl_sae.py`'s KL(rho||p_hat_f) constrains each feature's MARGINAL firing rate (averaged over the batch) but nothing constrains the JOINT distribution of active counts per sample -- at CIFAR-10, lambda=0.01, this let 57% of samples collapse to <5 active features while a dense minority absorbed the steering/reconstruction mass.

`rate_kl_joint_activity.py` adds a second, per-sample term penalizing each sample's own active count toward the target N*rho, alongside the existing per-feature term. Sweeps `lambda_joint` on CIFAR-10 (the dataset where bimodality actually occurred) at `lambda_kl=0.01` (the failing config), plus a Fashion-MNIST sanity check (bimodality was mild there, so this checks the fix doesn't hurt a config that wasn't broken).

**What to look for:** `lt5active` should drop toward 0 as `lambda_joint` increases (bimodality fixed) -- but watch `Top10Purity` and `Ablate` at the same time. The risk this experiment is designed to catch: enforcing BOTH per-feature rate AND per-sample count could make "same few big features, every sample" the easiest way to satisfy both constraints at once -- reintroducing the hub pathology Rate-KL was built to avoid. If purity collapses as lambda_joint rises, the fix works on paper but fails the paper's own central test.

**Before running:** Runtime -> Change runtime type -> GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go to Runtime > Change runtime type > GPU)')

In [ ]:
!git clone https://github.com/willkn/SAE-Gini.git
%cd SAE-Gini/experiments

## CIFAR-10 (the dataset that actually went bimodal), lambda_kl=0.01

In [ ]:
!python rate_kl_joint_activity.py --dataset cifar10 --seed 0 --rho 0.09 --lambda-kl 0.01 --lambda-joint 0.0 0.1 0.5 1.0 2.0

## Fashion-MNIST sanity check (bimodality was mild here -- fix shouldn't hurt)

In [ ]:
!python rate_kl_joint_activity.py --dataset fashion_mnist --seed 0 --rho 0.09 --lambda-kl 0.01 --lambda-joint 0.0 0.5 1.0

In [ ]:
import json, glob
for path in sorted(glob.glob('results/rate_kl_joint/*.json')):
    print(f'\n=== {path} ===')
    with open(path) as f:
        results = json.load(f)
    print(f"{'Model':32s} {'Sparsity':>9s} {'MSE':>8s} {'Top10Purity':>12s} {'Ablate':>8s} {'lt5active':>10s}")
    for name, r in results.items():
        ad = r['activity_distribution']
        print(f"{name:32s} {r['relative_sparsity']:9.3f} {r['mse']:8.4f} "
              f"{r['mean_purity_top10pct_by_importance']:12.3f} {r['steering_impact_ablate']:8.4f} "
              f"{ad['frac_samples_lt5_active']:10.3f}")

In [ ]:
!zip -r bimodality_fix_results.zip results/rate_kl_joint
from google.colab import files
files.download('bimodality_fix_results.zip')